In [17]:
import sys
sys.path.append("..")

# 1. Método de doble logística

In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import interact
from scipy.optimize import curve_fit
from pipeline.ingesta import cargar_indices_desde_bd
from utils.aplicar_whittaker import aplicar_whittaker_series

from pipeline.modulo_fenologico import segmentar_ciclos, detectar_sos
# Ajusta esta ruta al módulo real donde viven las funciones que pegaste
from pipeline.modulo_predictivo import (
    ajustar_curva_doble_logistica,
    extender_serie_con_curva_parametrica,
)

FECHA_INICIO = "2021-01-01"
FECHA_FIN = "2025-12-31"
DISTANCIA_MIN_DIAS = 90
PROMINENCIA_MIN = 0.15
FACTOR_SOS = 0.2
EOS_DIAS = 120  # SOS + 120, tu convención ya establecida

2026-07-07 23:41:13.161 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.163 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.164 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.165 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.166 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.167 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.168 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-07 23:41:13.169 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-07-

In [3]:
dfs_raw = cargar_indices_desde_bd(FECHA_INICIO, FECHA_FIN)
dfs_suave = aplicar_whittaker_series(dfs_raw, lambda_param=10000.0, orden=2)

print("Índices disponibles:", list(dfs_raw.keys()))
print("Parcelas:", dfs_raw["EVI"].columns.tolist())

✅  Índices cargados desde BD: 1826 fechas × 9 parcelas (2021-01-01 → 2025-12-31).
📈 Suavizando serie temporal para: EVI...
📈 Suavizando serie temporal para: LSWI...
Índices disponibles: ['EVI', 'LSWI']
Parcelas: ['id_0', 'id_1', 'id_2', 'id_3', 'id_4', 'id_5', 'id_6', 'id_7', 'id_8']


In [4]:
def obtener_ciclos_con_sos(parcela: str, indice_nombre: str = "EVI") -> list[dict]:
    """
    Replica la lógica de segmentación + detección de SOS de
    graficar_whittaker_sos, pero devuelve los resultados como estructura
    de datos en vez de graficarlos directamente. Cada ciclo detectado
    incluye sus límites (valles) y su fecha SOS si se pudo detectar.
    """
    serie = dfs_suave[indice_nombre][parcela]
    segmentos = segmentar_ciclos(serie, DISTANCIA_MIN_DIAS, PROMINENCIA_MIN)

    ciclos = []
    for inicio, fin in segmentos:
        es_ultimo = (inicio, fin) == segmentos[-1]
        mask = (serie.index >= inicio) & (serie.index <= fin) if not es_ultimo else (serie.index >= inicio)
        df_segmento = serie.loc[mask]

        if df_segmento.empty:
            continue

        resultado_sos = detectar_sos(
            serie=df_segmento.values,
            fechas=df_segmento.index,
            factor=FACTOR_SOS,
            metodo="seasonal_amplitude",
            ventana_busqueda=(inicio, serie.index.max()) if es_ultimo else (inicio, fin),
        )
        fecha_sos = resultado_sos.get("sos_fecha")
        if fecha_sos is None:
            continue

        ciclos.append({
            "inicio_valle": inicio,
            "fin_valle": fin,
            "fecha_sos": pd.Timestamp(fecha_sos),
            "fecha_eos": pd.Timestamp(fecha_sos) + pd.Timedelta(days=EOS_DIAS),
        })
    return ciclos

In [5]:
def simular_extrapolacion_ventana(
    parcela: str,
    indice_nombre: str,
    ciclo: dict,
    ventana: str,  # "T1", "T2", "T3"
) -> dict:
    """
    Trunca artificialmente la serie suavizada de un ciclo en el día
    correspondiente a la ventana (30/60/90 post-SOS), extrapola hasta EOS,
    y conserva también la serie real completa del ciclo para poder
    comparar visualmente la extrapolación contra lo que de verdad ocurrió.
    """
    dias_ventana = {"T1": 30, "T2": 60, "T3": 90}[ventana]
    fecha_sos = ciclo["fecha_sos"]
    fecha_eos = ciclo["fecha_eos"]
    fecha_corte = fecha_sos + pd.Timedelta(days=dias_ventana)

    serie_completa = dfs_suave[indice_nombre][parcela]
    serie_ciclo_real = serie_completa.loc[fecha_sos:fecha_eos]

    # "Lo que se hubiera conocido" hasta la ventana T simulada
    serie_truncada = serie_ciclo_real.loc[:fecha_corte]

    if fecha_corte not in serie_ciclo_real.index:
        return None  # el ciclo aún no llega a esta ventana, no se puede simular

    serie_extendida, params = extender_serie_con_curva_parametrica(
        serie_suavizada=serie_truncada,
        fecha_sos=fecha_sos,
        fecha_fin_extension=fecha_eos,
    )

    # Solo la porción extrapolada (posterior al corte), para graficarla aparte
    serie_solo_extrapolada = serie_extendida.loc[fecha_corte:]

    return {
        "fecha_corte": fecha_corte,
        "fecha_sos": fecha_sos,
        "fecha_eos": fecha_eos,
        "serie_truncada": serie_truncada,
        "serie_extrapolada": serie_solo_extrapolada,
        "serie_real_completa": serie_ciclo_real,
        "params": params,
    }

In [6]:
def graficar_validacion_extrapolacion(parcela: str, indice_nombre: str, ventana: str):
    ciclos = obtener_ciclos_con_sos(parcela, indice_nombre)
    if not ciclos:
        print("No se detectaron ciclos con SOS válido para esta parcela.")
        return

    opciones_ciclo = {
        f"SOS {c['fecha_sos'].date()} → EOS {c['fecha_eos'].date()}": c
        for c in ciclos
    }

    def actualizar(ciclo_label):
        ciclo = opciones_ciclo[ciclo_label]
        resultado = simular_extrapolacion_ventana(parcela, indice_nombre, ciclo, ventana)

        if resultado is None:
            print(f"Este ciclo no alcanza la ventana {ventana} (termina antes del día correspondiente).")
            return

        fig = go.Figure()

        # A. Serie real completa del ciclo (referencia, gris tenue de fondo)
        fig.add_trace(go.Scatter(
            x=resultado["serie_real_completa"].index,
            y=resultado["serie_real_completa"].values,
            mode="lines",
            line=dict(color="lightgray", width=2),
            name="Serie real completa (ground truth histórico)",
        ))

        # B. Porción "conocida" hasta la ventana T (sólido azul)
        fig.add_trace(go.Scatter(
            x=resultado["serie_truncada"].index,
            y=resultado["serie_truncada"].values,
            mode="lines",
            line=dict(color="#1f77b4", width=2.5),
            name=f"Suavizada observada hasta {ventana}",
        ))

        # C. Extrapolación: discontinua, empieza exactamente en el punto de corte
        serie_extrap = resultado["serie_extrapolada"]
        fig.add_trace(go.Scatter(
            x=serie_extrap.index,
            y=serie_extrap.values,
            mode="lines",
            line=dict(color="#d62728", width=2.5, dash="dash"),
            name="Extrapolación (curva doble logística)",
        ))

        # D. Líneas de referencia: SOS, corte de ventana, EOS
        fig.add_vline(x=ciclo["fecha_sos"], line_width=1.5, line_dash="dot", line_color="green")
        fig.add_annotation(x=ciclo["fecha_sos"], y=1, yref="paper", text="SOS", showarrow=False,
                            textangle=-90, xanchor="right", yanchor="top", font=dict(color="green", size=9))

        fig.add_vline(x=resultado["fecha_corte"], line_width=1.5, line_dash="dot", line_color="#d62728")
        fig.add_annotation(x=resultado["fecha_corte"], y=1, yref="paper", text=f"Corte {ventana}",
                            showarrow=False, textangle=-90, xanchor="right", yanchor="top",
                            font=dict(color="#d62728", size=9))

        fig.add_vline(x=ciclo["fecha_eos"], line_width=1.5, line_dash="dot", line_color="gray")
        fig.add_annotation(x=ciclo["fecha_eos"], y=1, yref="paper", text="EOS (SOS+120)",
                            showarrow=False, textangle=-90, xanchor="right", yanchor="top",
                            font=dict(color="gray", size=9))

        # E. Métrica de calidad del ajuste en el título
        params = resultado["params"]
        r2_txt = f"R²={params['r2']:.3f}" if params is not None else "ajuste no confiable (sin extrapolar)"

        fig.update_layout(
            title=dict(
                text=f"Validación extrapolación: {indice_nombre} en {parcela} — Ventana {ventana} ({r2_txt})",
                font=dict(size=14, weight="bold"),
            ),
            xaxis_title="Fecha",
            yaxis_title=indice_nombre,
            template="plotly_white",
            hovermode="x unified",
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
            width=1100,
            height=550,
            margin=dict(t=90, b=40, l=50, r=40),
        )
        fig.show()

    interact(
        actualizar,
        ciclo_label=widgets.Dropdown(options=list(opciones_ciclo.keys()), description="Ciclo:"),
    )

In [ ]:
graficar_validacion_extrapolacion(
    parcela="id_3",       # ajusta al id real de tu parcela
    indice_nombre="EVI",  # o "LSWI"
    ventana="T2",         # "T1", "T2" o "T3"
)

interactive(children=(Dropdown(description='Ciclo:', options=('SOS 2022-08-17 → EOS 2022-12-15', 'SOS 2023-04-…

In [10]:
def calcular_fecha_fin_extension_segura(
    ciclo: dict,
    ciclos_ordenados: list[dict],
    margen_dias: int = 10,
) -> pd.Timestamp:
    """
    Determina la fecha límite de extrapolación para un ciclo, capando
    SOS+120 si el siguiente ciclo detectado arranca antes de esa fecha.
    Evita que la curva doble logística intente modelar el arranque del
    siguiente ciclo, para el cual no tiene forma funcional adecuada.
    """
    idx = ciclos_ordenados.index(ciclo)
    eos_teorico = ciclo["fecha_eos"]  # SOS + 120

    if idx + 1 < len(ciclos_ordenados):
        siguiente_sos = ciclos_ordenados[idx + 1]["fecha_sos"]
        limite_por_contaminacion = siguiente_sos - pd.Timedelta(days=margen_dias)
        return min(eos_teorico, limite_por_contaminacion)

    return eos_teorico

In [11]:
from pipeline.modulo_predictivo import _doble_logistica
def evaluar_plausibilidad_extrapolacion(
    serie_truncada: pd.Series,
    params: dict,
    fecha_sos: pd.Timestamp,
    dias_ventana_tendencia: int = 7,
) -> dict:
    """
    Compara la tendencia local observada (últimos N días antes del corte)
    contra la tendencia que implica la curva ajustada en ese mismo punto.
    Una discrepancia fuerte de signo indica que el modelo unimodal no
    captura la dinámica real (ej. inicio de un nuevo ciclo bleeding-in),
    aunque el R² del ajuste sobre los datos observados sea alto.
    """
    y_recientes = serie_truncada.iloc[-dias_ventana_tendencia:].to_numpy()
    x_recientes = np.arange(len(y_recientes), dtype=float)
    pendiente_observada = np.polyfit(x_recientes, y_recientes, 1)[0]

    ultimo_dia = (serie_truncada.index[-1] - fecha_sos).days
    delta = 1.0
    y0 = _doble_logistica(ultimo_dia, **{k: params[k] for k in ["vmin","vmax","S","mS","A","mA"]})
    y1 = _doble_logistica(ultimo_dia + delta, **{k: params[k] for k in ["vmin","vmax","S","mS","A","mA"]})
    pendiente_modelo = (y1 - y0) / delta

    signos_coinciden = np.sign(pendiente_observada) == np.sign(pendiente_modelo)
    magnitud_razonable = abs(pendiente_modelo) < 3 * abs(pendiente_observada) + 1e-3

    return {
        "pendiente_observada": pendiente_observada,
        "pendiente_modelo": pendiente_modelo,
        "plausible": bool(signos_coinciden and magnitud_razonable),
    }

In [12]:
def validar_extrapolaciones_historicas(
    parcelas: list[str],
    indice_nombre: str,
    ventanas: list[str] = ["T1", "T2", "T3"],
) -> pd.DataFrame:
    """
    Recorre todos los ciclos históricos de las parcelas dadas y calcula
    el error de la extrapolación contra la serie real observada (que en
    modo histórico ya se conoce), para cada ventana T. Permite detectar
    sistemáticamente los casos donde el modelo doble-logístico falla
    (ej. contaminación por ciclo siguiente), en vez de depender de
    inspección visual caso por caso.
    """
    filas = []
    for parcela in parcelas:
        ciclos = obtener_ciclos_con_sos(parcela, indice_nombre)
        ciclos_ordenados = sorted(ciclos, key=lambda c: c["fecha_sos"])

        for ciclo in ciclos_ordenados:
            fecha_eos_segura = calcular_fecha_fin_extension_segura(ciclo, ciclos_ordenados)

            for ventana in ventanas:
                resultado = simular_extrapolacion_ventana(parcela, indice_nombre, ciclo, ventana)
                if resultado is None or resultado["params"] is None:
                    continue

                plausibilidad = evaluar_plausibilidad_extrapolacion(
                    resultado["serie_truncada"], resultado["params"], ciclo["fecha_sos"]
                )

                # Comparar extrapolación vs realidad SOLO dentro de la ventana segura
                real_post_corte = resultado["serie_real_completa"].loc[
                    resultado["fecha_corte"]:fecha_eos_segura
                ]
                extrap_post_corte = resultado["serie_extrapolada"].loc[
                    :fecha_eos_segura
                ].reindex(real_post_corte.index)

                error = (extrap_post_corte - real_post_corte).dropna()
                rmse = np.sqrt((error ** 2).mean()) if len(error) > 0 else np.nan

                filas.append({
                    "parcela": parcela,
                    "sos": ciclo["fecha_sos"],
                    "ventana": ventana,
                    "r2_ajuste": resultado["params"]["r2"],
                    "rmse_extrapolacion": rmse,
                    "plausible": plausibilidad["plausible"],
                })

    return pd.DataFrame(filas)

In [13]:
df_validacion = validar_extrapolaciones_historicas(
    parcelas=dfs_suave["EVI"].columns.tolist(),
    indice_nombre="EVI",
)
df_validacion.sort_values("rmse_extrapolacion", ascending=False).head(15)

,parcela,sos,ventana,r2_ajuste,rmse_extrapolacion,plausible
162,id_8,2025-05-08,T2,0.999587,0.677324,False
64,id_2,2022-10-04,T2,0.999858,0.672946,True
85,id_3,2024-05-26,T2,1.000000,0.382630,True
26,id_0,2025-04-07,T3,0.998654,0.333292,False
59,id_1,2025-03-25,T3,0.999590,0.310025,True
118,id_6,2022-06-27,T2,0.999997,0.224478,True
21,id_0,2024-09-26,T1,0.999999,0.208955,True
36,id_1,2022-05-28,T1,0.999999,0.205551,True
12,id_0,2023-03-02,T1,1.000000,0.204612,True
112,id_5,2024-06-20,T2,0.999999,0.203384,True


# 2. Método Whittaker + Climatología

In [30]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import interact
from whittaker_eilers import WhittakerSmoother

from pipeline.modulo_fenologico import segmentar_ciclos, detectar_sos
from pipeline.modulo_predictivo import (
    ajustar_curva_doble_logistica,
    extender_serie_con_curva_parametrica,
    obtener_climatologia
)

FECHA_INICIO = "2021-01-01"
FECHA_FIN = "2025-12-31"
DISTANCIA_MIN_DIAS = 90
PROMINENCIA_MIN = 0.15
FACTOR_SOS = 0.2
EOS_DIAS = 120
LAMBDA_WHITTAKER = 10000.0
ORDEN_WHITTAKER = 2

In [31]:
dfs_raw = cargar_indices_desde_bd(FECHA_INICIO, FECHA_FIN)
dfs_suave = aplicar_whittaker_series(dfs_raw, lambda_param=LAMBDA_WHITTAKER, orden=ORDEN_WHITTAKER)


def obtener_ciclos_con_sos(parcela: str, indice_nombre: str = "EVI") -> list[dict]:
    """
    Segmenta la serie suavizada en ciclos y detecta SOS por ciclo,
    reutilizando la lógica ya validada de graficar_whittaker_sos.
    """
    serie = dfs_suave[indice_nombre][parcela]
    segmentos = segmentar_ciclos(serie, DISTANCIA_MIN_DIAS, PROMINENCIA_MIN)

    ciclos = []
    for inicio, fin in segmentos:
        es_ultimo = (inicio, fin) == segmentos[-1]
        mask = (serie.index >= inicio) & (serie.index <= fin) if not es_ultimo else (serie.index >= inicio)
        df_segmento = serie.loc[mask]
        if df_segmento.empty:
            continue

        resultado_sos = detectar_sos(
            serie=df_segmento.values,
            fechas=df_segmento.index,
            factor=FACTOR_SOS,
            metodo="seasonal_amplitude",
            ventana_busqueda=(inicio, serie.index.max()) if es_ultimo else (inicio, fin),
        )
        fecha_sos = resultado_sos.get("sos_fecha")
        if fecha_sos is None:
            continue

        fecha_sos = pd.Timestamp(fecha_sos)
        ciclos.append({
            "inicio_valle": inicio,
            "fin_valle": fin,
            "fecha_sos": fecha_sos,
            "fecha_eos": fecha_sos + pd.Timedelta(days=EOS_DIAS),
        })
    return sorted(ciclos, key=lambda c: c["fecha_sos"])

✅  Índices cargados desde BD: 1826 fechas × 9 parcelas (2021-01-01 → 2025-12-31).
📈 Suavizando serie temporal para: EVI...
📈 Suavizando serie temporal para: LSWI...


In [32]:
def whittaker_con_pesos_personalizados(
    serie_valores: np.ndarray,
    pesos: np.ndarray,
    lambda_param: float,
    orden: int = ORDEN_WHITTAKER,
) -> np.ndarray:
    """
    Variante de una sola serie de aplicar_whittaker_series, que acepta
    un vector de pesos y un lambda arbitrarios (no atados al lambda del
    suavizado principal), necesarios porque las anclas climatológicas
    tienen peso « 1 y requieren una penalización de rugosidad mucho
    menor para que la curva realmente se doble hacia ellas.
    """
    valores_preparados = np.nan_to_num(serie_valores, nan=0.0).tolist()
    num_validos = int(np.sum(pesos > 0))

    if num_validos < (orden + 1):
        return pd.Series(serie_valores).interpolate(
            method="linear", limit_direction="both"
        ).fillna(0.0).to_numpy()

    whittaker = WhittakerSmoother(
        lmbda=lambda_param,
        order=orden,
        data_length=len(valores_preparados),
        x_input=None,
        weights=pesos.tolist(),
    )
    return np.array(whittaker.smooth(valores_preparados))

In [42]:
LAMBDA_EXTRAPOLACION = 300.0   # deliberadamente << LAMBDA_WHITTAKER (10000)
PESO_INICIAL_ANCLA = 5.0       # peso fuerte para forzar continuidad exacta en el corte
PESO_CLIMATOLOGIA = 0.3

def construir_segmento_extrapolado_climatologia(
    serie_truncada: pd.Series,
    fecha_fin_extension: pd.Timestamp,
    climatologia: pd.Series,      # <-- ahora se recibe precalculada, no se consulta a BD
    peso_inicial: float = PESO_INICIAL_ANCLA,
    peso_ancla: float = PESO_CLIMATOLOGIA,
    espaciado_dias: int = 1,
    lambda_extrapolacion: float = LAMBDA_EXTRAPOLACION,
) -> pd.Series:
    """
    (sin cambios en la lógica interna — solo cambia el origen del dato:
    climatologia ahora viene de calcular_climatologia_indice_espectral,
    no de obtener_climatologia/climatologia_diaria)
    """
    fechas = pd.date_range(serie_truncada.index[-1], fecha_fin_extension, freq="D")
    valores = np.full(len(fechas), np.nan)
    pesos = np.zeros(len(fechas))

    valores[0] = serie_truncada.iloc[-1]
    pesos[0] = peso_inicial

    for i in range(1, len(fechas)):
        if i % espaciado_dias != 0:
            continue
        dia_anio = fechas[i].dayofyear
        v = climatologia.get(dia_anio, np.nan)
        if not np.isnan(v):
            valores[i] = v
            pesos[i] = peso_ancla

    valores_suaves = whittaker_con_pesos_personalizados(
        valores, pesos, lambda_param=lambda_extrapolacion
    )
    return pd.Series(valores_suaves, index=fechas)

In [43]:
def simular_extrapolacion_comparativa(
    parcela: str,
    indice_nombre: str,
    ciclo: dict,
    ventana: str,
    climatologia: pd.Series,   # <-- se pasa ya calculada
    peso_ancla: float = PESO_CLIMATOLOGIA,
    espaciado_dias: int = 1,
    lambda_extrapolacion: float = LAMBDA_EXTRAPOLACION,
) -> dict | None:
    dias_ventana = {"T1": 30, "T2": 60, "T3": 90}[ventana]
    fecha_sos = ciclo["fecha_sos"]
    fecha_eos = ciclo["fecha_eos"]
    fecha_corte = fecha_sos + pd.Timedelta(days=dias_ventana)

    serie_completa_real = dfs_suave[indice_nombre][parcela]
    serie_ciclo_real = serie_completa_real.loc[fecha_sos:fecha_eos]

    if fecha_corte not in serie_ciclo_real.index:
        return None

    serie_truncada = serie_ciclo_real.loc[:fecha_corte]

    serie_extendida_dl, params_dl = extender_serie_con_curva_parametrica(
        serie_suavizada=serie_truncada, fecha_sos=fecha_sos, fecha_fin_extension=fecha_eos,
    )
    extrap_dl = serie_extendida_dl.loc[fecha_corte:]

    extrap_clima = construir_segmento_extrapolado_climatologia(
        serie_truncada=serie_truncada,
        fecha_fin_extension=fecha_eos,
        climatologia=climatologia,
        peso_ancla=peso_ancla,
        espaciado_dias=espaciado_dias,
        lambda_extrapolacion=lambda_extrapolacion,
    )

    return {
        "fecha_corte": fecha_corte, "fecha_sos": fecha_sos, "fecha_eos": fecha_eos,
        "serie_truncada": serie_truncada, "serie_real_completa": serie_ciclo_real,
        "extrap_doble_logistica": extrap_dl, "extrap_climatologia": extrap_clima,
        "params_dl": params_dl,
    }

In [35]:
def graficar_comparacion_metodos(parcela: str, indice_nombre: str, ventana: str, id_region: int = 1):
    ciclos = obtener_ciclos_con_sos(parcela, indice_nombre)
    if not ciclos:
        print("No se detectaron ciclos con SOS válido para esta parcela.")
        return

    opciones_ciclo = {
        f"SOS {c['fecha_sos'].date()} → EOS {c['fecha_eos'].date()}": c
        for c in ciclos
    }

    def actualizar(ciclo_label):
        ciclo = opciones_ciclo[ciclo_label]
        r = simular_extrapolacion_comparativa(parcela, indice_nombre, ciclo, ventana, id_region=id_region)

        if r is None:
            print(f"Este ciclo no alcanza la ventana {ventana}.")
            return

        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=r["serie_real_completa"].index, y=r["serie_real_completa"].values,
            mode="lines", line=dict(color="lightgray", width=2),
            name="Serie real completa (ground truth histórico)",
        ))

        fig.add_trace(go.Scatter(
            x=r["serie_truncada"].index, y=r["serie_truncada"].values,
            mode="lines", line=dict(color="#1f77b4", width=2.5),
            name=f"Suavizada observada hasta {ventana}",
        ))

        fig.add_trace(go.Scatter(
            x=r["extrap_doble_logistica"].index, y=r["extrap_doble_logistica"].values,
            mode="lines", line=dict(color="#d62728", width=2.5, dash="dash"),
            name="Extrapolación: doble logística",
        ))

        fig.add_trace(go.Scatter(
            x=r["extrap_climatologia"].index, y=r["extrap_climatologia"].values,
            mode="lines", line=dict(color="#2ca02c", width=2.5, dash="dash"),
            name="Extrapolación: climatología anclada + Whittaker",
        ))

        for fecha, color, texto in [
            (ciclo["fecha_sos"], "green", "SOS"),
            (r["fecha_corte"], "#d62728", f"Corte {ventana}"),
            (ciclo["fecha_eos"], "gray", "EOS (SOS+120)"),
        ]:
            fig.add_vline(x=fecha, line_width=1.5, line_dash="dot", line_color=color)
            fig.add_annotation(x=fecha, y=1, yref="paper", text=texto, showarrow=False,
                                textangle=-90, xanchor="right", yanchor="top",
                                font=dict(color=color, size=9))

        r2_txt = f"R² doble-log={r['params_dl']['r2']:.3f}" if r["params_dl"] else "doble-log no convergió"

        fig.update_layout(
            title=dict(
                text=f"Comparación de extrapolación: {indice_nombre} en {parcela} — {ventana} ({r2_txt})",
                font=dict(size=14, weight="bold"),
            ),
            xaxis_title="Fecha", yaxis_title=indice_nombre,
            template="plotly_white", hovermode="x unified",
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
            width=1150, height=550, margin=dict(t=90, b=40, l=50, r=40),
        )
        fig.show()

    interact(
        actualizar,
        ciclo_label=widgets.Dropdown(options=list(opciones_ciclo.keys()), description="Ciclo:"),
    )

In [ ]:
graficar_comparacion_metodos(
    parcela="id_1",
    indice_nombre="EVI",
    ventana="T1",
    id_region=1,
)

interactive(children=(Dropdown(description='Ciclo:', options=('SOS 2021-07-22 → EOS 2021-11-19', 'SOS 2021-10-…

In [37]:
def diagnosticar_lambda_extrapolacion(parcela: str, indice_nombre: str, ventana: str, id_region: int = 1):
    ciclos = obtener_ciclos_con_sos(parcela, indice_nombre)
    ciclo = ciclos[-1]  # usa el último ciclo disponible como caso de prueba

    dias_ventana = {"T1": 30, "T2": 60, "T3": 90}[ventana]
    fecha_sos = ciclo["fecha_sos"]
    fecha_eos = ciclo["fecha_eos"]
    fecha_corte = fecha_sos + pd.Timedelta(days=dias_ventana)

    serie_ciclo_real = dfs_suave[indice_nombre][parcela].loc[fecha_sos:fecha_eos]
    serie_truncada = serie_ciclo_real.loc[:fecha_corte]

    def actualizar(log10_lambda, peso_ancla):
        lam = 10 ** log10_lambda
        extrap = construir_segmento_extrapolado_climatologia(
            serie_truncada=serie_truncada,
            fecha_fin_extension=fecha_eos,
            variable_climatologia=indice_nombre,
            id_region=id_region,
            peso_ancla=peso_ancla,
            lambda_extrapolacion=lam,
        )
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=serie_ciclo_real.index, y=serie_ciclo_real.values,
                                  mode="lines", line=dict(color="lightgray", width=2), name="Real completa"))
        fig.add_trace(go.Scatter(x=serie_truncada.index, y=serie_truncada.values,
                                  mode="lines", line=dict(color="#1f77b4", width=2.5), name="Observada"))
        fig.add_trace(go.Scatter(x=extrap.index, y=extrap.values,
                                  mode="lines", line=dict(color="#2ca02c", width=2.5, dash="dash"),
                                  name=f"Extrapolación (λ={lam:.2f}, peso_ancla={peso_ancla})"))
        fig.update_layout(template="plotly_white", height=450, width=1000,
                           title=f"λ={lam:.2f} | peso_ancla={peso_ancla}")
        fig.show()

    interact(
        actualizar,
        log10_lambda=widgets.FloatSlider(min=-2, max=3, step=0.25, value=1, description="log10(λ)"),
        peso_ancla=widgets.FloatSlider(min=0.05, max=2.0, step=0.05, value=0.3, description="peso ancla"),
    )

In [38]:
diagnosticar_lambda_extrapolacion(parcela="id_1", indice_nombre="EVI", ventana="T3")

interactive(children=(FloatSlider(value=1.0, description='log10(λ)', max=3.0, min=-2.0, step=0.25), FloatSlide…

In [39]:
def diagnosticar_extrapolacion_paso_a_paso(parcela: str, indice_nombre: str, ventana: str, id_region: int = 1):
    ciclos = obtener_ciclos_con_sos(parcela, indice_nombre)
    ciclo = ciclos[-1]

    dias_ventana = {"T1": 30, "T2": 60, "T3": 90}[ventana]
    fecha_sos = ciclo["fecha_sos"]
    fecha_eos = ciclo["fecha_eos"]
    fecha_corte = fecha_sos + pd.Timedelta(days=dias_ventana)

    serie_ciclo_real = dfs_suave[indice_nombre][parcela].loc[fecha_sos:fecha_eos]
    serie_truncada = serie_ciclo_real.loc[:fecha_corte]

    # --- PASO 1: ¿la climatología devuelve datos reales? ---
    climatologia = obtener_climatologia(indice_nombre, id_region)
    print(f"[1] Climatología '{indice_nombre}' región {id_region}: {len(climatologia)} filas")
    print(f"    dia_anio min/max: {climatologia.index.min()} / {climatologia.index.max()}")
    print(f"    valores NaN: {climatologia.isna().sum()} de {len(climatologia)}")
    print(climatologia.head(3), "\n")

    if climatologia.empty or climatologia.isna().all():
        print("❌ PROBLEMA ENCONTRADO: climatología vacía o toda NaN.")
        print("   Revisa: nombre de 'variable' en la tabla (¿coincide 'EVI' exacto?), id_region correcto.")
        return

    # --- PASO 2: construir el vector de anclas manualmente e inspeccionarlo ---
    fechas = pd.date_range(serie_truncada.index[-1], fecha_eos, freq="D")
    valores = np.full(len(fechas), np.nan)
    pesos = np.zeros(len(fechas))
    valores[0] = serie_truncada.iloc[-1]
    pesos[0] = PESO_INICIAL_ANCLA

    aciertos, fallos = 0, 0
    for i in range(1, len(fechas)):
        dia_anio = fechas[i].dayofyear
        v = climatologia.get(dia_anio, np.nan)
        if not np.isnan(v):
            valores[i] = v
            pesos[i] = PESO_CLIMATOLOGIA
            aciertos += 1
        else:
            fallos += 1

    print(f"[2] Anclas climatológicas colocadas: {aciertos} aciertos, {fallos} fallos (NaN)")
    num_validos = int(np.sum(pesos > 0))
    print(f"    Total puntos con peso > 0: {num_validos}  (orden+1 = {ORDEN_WHITTAKER + 1})")

    if num_validos < (ORDEN_WHITTAKER + 1):
        print("❌ PROBLEMA ENCONTRADO: menos puntos válidos que orden+1.")
        print("   → whittaker_con_pesos_personalizados está cayendo en el fallback de")
        print("     interpolación lineal, que ignora lambda por completo. Esto explica")
        print("     que la curva no cambie sin importar qué λ pruebes.")
        return

    # --- PASO 3: llamar dos veces con lambdas extremos y comparar directamente ---
    salida_lambda_bajo = whittaker_con_pesos_personalizados(valores, pesos, lambda_param=0.1)
    salida_lambda_alto = whittaker_con_pesos_personalizados(valores, pesos, lambda_param=10000)

    diferencia_max = np.max(np.abs(salida_lambda_bajo - salida_lambda_alto))
    print(f"[3] Diferencia máxima entre λ=0.1 y λ=10000: {diferencia_max:.6f}")

    if diferencia_max < 1e-6:
        print("❌ PROBLEMA ENCONTRADO: la salida es IDÉNTICA sin importar λ.")
        print("   El WhittakerSmoother no está reaccionando al parámetro lambda_param.")
        print("   Posibles causas: la librería requiere otro nombre de argumento,")
        print("   o el objeto se está cacheando/reutilizando mal.")
    else:
        print("✅ El suavizador SÍ reacciona a λ en este test aislado.")
        print("   Si en tu gráfico completo sigues viendo una recta, el problema")
        print("   probablemente está en cómo simular_extrapolacion_comparativa")
        print("   está llamando a la función (revisa que no esté hardcodeado un λ fijo,")
        print("   o que el resultado no se esté sobrescribiendo después).")

    return valores, pesos, salida_lambda_bajo, salida_lambda_alto

In [40]:
resultado_diag = diagnosticar_extrapolacion_paso_a_paso(
    parcela="id_1", indice_nombre="EVI", ventana="T3"
)

[1] Climatología 'EVI' región 1: 0 filas
    dia_anio min/max: nan / nan
    valores NaN: 0 de 0
Series([], Name: valor_climatologico, dtype: object) 

❌ PROBLEMA ENCONTRADO: climatología vacía o toda NaN.
   Revisa: nombre de 'variable' en la tabla (¿coincide 'EVI' exacto?), id_region correcto.


In [41]:
def calcular_climatologia_indice_espectral(
    indice_nombre: str,
    parcelas: list[str] | None = None,
    metodo_agregacion: str = "median",  # "median" es más robusto a outliers que "mean"
) -> pd.Series:
    """
    Calcula una climatología diaria (por dia_anio, 1-366) de un índice
    espectral (EVI/LSWI) a partir de las series suavizadas históricas
    ya cargadas en memoria (dfs_suave), agregando across años y,
    opcionalmente, across parcelas.

    NOTA: climatologia_diaria (la tabla de BD) está reservada por CHECK
    constraint a variables meteorológicas ('PAR', 'temperatura'), por lo
    que la climatología de índices espectrales se calcula en memoria,
    análogo al patrón de cold-start ya usado para LSWI_max con el
    archivo histórico crudo de Sentinel-2.

    Parámetros
    ----------
    parcelas : list[str] | None
        Si es None, agrega usando todas las parcelas disponibles en
        dfs_suave (climatología regional). Si se especifica, agrega solo
        esas parcelas (climatología intra-parcela, útil si sospechas
        heterogeneidad fuerte de suelo/manejo entre parcelas).

    Retorna
    -------
    pd.Series
        Índice = dia_anio (1-366), valor = índice espectral agregado.
    """
    df = dfs_suave[indice_nombre]
    columnas = parcelas if parcelas is not None else df.columns.tolist()

    df_dia_anio = df[columnas].copy()
    df_dia_anio["dia_anio"] = df_dia_anio.index.dayofyear

    if metodo_agregacion == "median":
        climatologia = df_dia_anio.groupby("dia_anio")[columnas].median().mean(axis=1)
    else:
        climatologia = df_dia_anio.groupby("dia_anio")[columnas].mean().mean(axis=1)

    climatologia.name = "valor_climatologico"
    return climatologia

In [44]:
climatologia_evi = calcular_climatologia_indice_espectral("EVI")
print(f"Climatología EVI: {len(climatologia_evi)} días con dato, rango dia_anio {climatologia_evi.index.min()}-{climatologia_evi.index.max()}")
climatologia_evi.head()

Climatología EVI: 366 días con dato, rango dia_anio 1-366


dia_anio
1    0.347320
2    0.343342
3    0.339467
4    0.335652
5    0.331902
Name: valor_climatologico, dtype: float64

In [45]:
resultado_diag = diagnosticar_extrapolacion_paso_a_paso(
    parcela="id_1", indice_nombre="EVI", ventana="T3"
)
# (ajusta la función de diagnóstico para recibir climatologia_evi en vez de llamar obtener_climatologia)

[1] Climatología 'EVI' región 1: 0 filas
    dia_anio min/max: nan / nan
    valores NaN: 0 de 0
Series([], Name: valor_climatologico, dtype: object) 

❌ PROBLEMA ENCONTRADO: climatología vacía o toda NaN.
   Revisa: nombre de 'variable' en la tabla (¿coincide 'EVI' exacto?), id_region correcto.
